In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import ast
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import ast


In [2]:

PROJECT_DIR = Path.cwd().parent
SCRIPTS_DIR = PROJECT_DIR / "scripts"
DATA_PROCESSED_DIR = PROJECT_DIR / "data_processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from topics_lda import (
    LDAToxicConfig,
    fit_lda_on_tokens,
    topics_as_dataframe,
    dominant_topic_per_doc,
)


In [3]:
df = pd.read_csv(DATA_PROCESSED_DIR / "reddit_with_toxicity.csv")

# Reconstruir listas desde CSV
for col in ["tokens", "tokens_no_stop"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df.head()


,body,created_utc,score,created_dt,date,year,month,year_month,text_clean,tokens,...,vad_arousal_mean,vad_dominance_mean,tox_word_count,tox_ratio,tox_has_toxic,tox_adv_token_count,tox_adv_score_raw,tox_adv_score_norm,tox_adv_max_span_score,tox_adv_has_toxic
0,Done - your wish is our command. Graves shall...,2017-04-01 16:01:43,2022,2017-04-01 16:01:43,2017-04-01,2017,4,2017-04,done your wish is our command graves shall onc...,"[done, your, wish, is, our, command, graves, s...",...,0.109800,0.097600,0,0.0,0,0,0.0,0.0,0.0,False
1,I don't play league but goddam it if I'm not g...,2017-04-01 03:56:12,4183,2017-04-01 03:56:12,2017-04-01,2017,4,2017-04,i don t play league but goddam it if i m not g...,"[i, don, t, play, league, but, goddam, it, if,...",...,0.068222,0.133444,0,0.0,0,0,0.0,0.0,0.0,False
2,http://imgur.com/a/8F7c6 WE DID IT REDDIT!,2017-04-01 04:01:06,267,2017-04-01 04:01:06,2017-04-01,2017,4,2017-04,http imgur com a 8f7c6 we did it reddit,"[http, imgur, com, a, 8f7c6, we, did, it, reddit]",...,0.667000,0.000000,0,0.0,0,0,0.0,0.0,0.0,False
3,Seems legit,2017-04-01 00:55:50,978,2017-04-01 00:55:50,2017-04-01,2017,4,2017-04,seems legit,"[seems, legit]",...,-0.146000,0.698000,0,0.0,0,0,0.0,0.0,0.0,False
4,"Graves no cigar, graves in peril, not to worry...",2017-04-01 01:19:17,670,2017-04-01 01:19:17,2017-04-01,2017,4,2017-04,graves no cigar graves in peril not to worry w...,"[graves, no, cigar, graves, in, peril, not, to...",...,0.432667,-0.246000,0,0.0,0,0,0.0,0.0,0.0,False


In [4]:
threshold = 0.05

df_toxic = df[df["tox_adv_score_norm"] >= threshold].copy()
print("Comentarios totales:", len(df))
print("Comentarios tóxicos (thr=0.05):", len(df_toxic))

# Checar un par
df_toxic[["body", "tox_adv_score_norm"]].head(10)


Comentarios totales: 26259
Comentarios tóxicos (thr=0.05): 2145


,body,tox_adv_score_norm
20,WE FUCKING DID IT BOYS,0.933333
39,I don't play League or even watch it but Fuck ...,0.500000
45,Holy shit it happened,0.666667
62,Holy shit 3hrs and it's here,0.666667
67,Holy shit it was true,0.666667
72,oh shit waddap\n,0.666667
74,LET'S FUCKING GO BBY!,0.933333
101,Thats insane mate Holy fuck!,0.500000
122,Holy fuck dude this looks amazing!! Great work...,0.312500
123,So fucking epic my goodness,0.933333


In [6]:
docs_tokens = df_toxic["tokens_no_stop"].tolist()
len(docs_tokens), docs_tokens[0][:20]


(2145, ['fucking', 'did', 'boys'])

In [7]:
from topics_lda import LDAToxicConfig, fit_lda_on_tokens

cfg = LDAToxicConfig(
    n_topics=8,      # puedes probar 5, 8, 10...
    max_features=5000,
    min_df=20,
    max_df=0.7,
    random_state=42,
)

lda_model, vectorizer, dt_matrix = fit_lda_on_tokens(docs_tokens, cfg)


In [8]:
topics_df = topics_as_dataframe(lda_model, vectorizer, top_n=15)
topics_df.head(30)


,topic_id,rank,term,weight
0,0,1,shit,126.437923
1,0,2,game,122.415384
2,0,3,like,99.763481
3,0,4,play,96.920196
4,0,5,playing,80.937370
5,0,6,just,66.095704
6,0,7,games,43.687749
7,0,8,league,43.085918
8,0,9,ranked,31.124766
9,0,10,hard,28.156540


In [10]:
topics_df.to_csv(OUTPUT_DIR / "lda_toxic_topics.csv", index=False)
print(" Guardado lda_toxic_topics.csv")


 Guardado lda_toxic_topics.csv


In [11]:
topics_pivot = (
    topics_df.pivot(index="topic_id", columns="rank", values="term")
    .reset_index()
)
topics_pivot


rank,topic_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0,shit,game,like,play,playing,just,games,league,ranked,hard,stop,time,ll,season,fun
1,1,fucking,riot,stupid,just,way,league,like,make,did,really,game,legends,skins,skin,let
2,2,man,fuck,t1,team,lol,fucking,faker,na,sad,shit,guma,org,gonna,just,honestly
3,3,fucking,fuck,good,oh,actually,god,time,actual,got,like,wait,riot,damn,did,imagine
4,4,fuck,trash,just,talk,game,hope,say,like,win,going,want,make,did,guy,doesn
5,5,just,lmao,troll,useless,garbage,yasuo,doing,play,player,int,guy,jungle,people,uninstall,toxic
6,6,shit,holy,seen,fucking,good,looks,awesome,thats,amazing,cool,series,dude,love,year,best
7,7,people,shit,like,just,game,players,know,really,games,think,play,bitch,lane,adc,enemy


In [12]:
from topics_lda import dominant_topic_per_doc

df_toxic["topic_id"] = dominant_topic_per_doc(lda_model, dt_matrix)
df_toxic[["body", "tox_adv_score_norm", "topic_id"]].head()


,body,tox_adv_score_norm,topic_id
20,WE FUCKING DID IT BOYS,0.933333,1
39,I don't play League or even watch it but Fuck ...,0.500000,4
45,Holy shit it happened,0.666667,6
62,Holy shit 3hrs and it's here,0.666667,6
67,Holy shit it was true,0.666667,6


In [13]:
df_toxic.to_csv(DATA_PROCESSED_DIR / "reddit_toxic_with_topics.csv", index=False)
print(" Guardado reddit_toxic_with_topics.csv")


 Guardado reddit_toxic_with_topics.csv


In [16]:
topic_year = (
    df_toxic.groupby(["year", "topic_id"])
            .size()
            .reset_index(name="count")
)

topic_year.head()


,year,topic_id,count
0,2015,0,2
1,2015,1,5
2,2015,2,2
3,2015,3,2
4,2015,4,8


In [17]:
topic_year.to_csv(OUTPUT_DIR / "lda_toxic_topics_by_year.csv", index=False)
print(" Guardado lda_toxic_topics_by_year.csv")


 Guardado lda_toxic_topics_by_year.csv
